In [1]:
import pandas as pd
import mlflow
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import os
from dotenv import load_dotenv
from sklearn.linear_model import LogisticRegression

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [3]:
load_dotenv()
path = os.getenv("DATA_PATH")
transaction_path = os.path.join(path, r'raw/train_transaction.csv/train_transaction.csv')
identity_path = os.path.join(path, r'raw/train_identity.csv/train_identity.csv')
train_transaction = pd.read_csv(transaction_path)
train_identity = pd.read_csv(identity_path)

In [4]:
df = train_transaction.merge(train_identity, on="TransactionID", how = "left")
df = df.sort_values("TransactionDT").reset_index(drop=True)

length = len(df)
train_end = int(length * 0.70)
val_end = int(length * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

assert train_df["TransactionDT"].max() < val_df["TransactionDT"].min()
assert val_df["TransactionDT"].max() < test_df["TransactionDT"].min()

In [5]:
base_features = ['TransactionDT', "TransactionAmt", "ProductCD", "P_emaildomain", "R_emaildomain"]
extended_features = base_features + ["card2", "card4", "card6", "addr1", "addr2", "dist1", "dist2"]
symbols = ['V', 'D', 'M', 'C', 'id', "card"]
def custom_features(df):

    df_custom = df[extended_features].copy()

    df_custom['missing_count'] = df.isna().sum(axis=1)

    for i in symbols:
        df_custom[f'missing_{i}_count'] = df.filter(regex = f"^{i}").isna().sum(axis=1)

    df_custom["transaction_day"] = df["TransactionDT"] // 86400

    df_custom["transaction_hour"] = (df["TransactionDT"] % 86400) // 3600

    return df_custom

In [6]:
feature_sets = {
    "baseline": base_features,
    "transaction": extended_features,
    "engineered": list(custom_features(train_df).columns)
}

In [7]:
X_train_sets = {
    "baseline": train_df[base_features],
    "transaction": train_df[extended_features],
    "engineered": custom_features(train_df)
}

X_val_sets = {
    "baseline": val_df[base_features],
    "transaction": val_df[extended_features],
    "engineered": custom_features(val_df)
}

X_test_sets = {
    "baseline": test_df[base_features],
    "transaction": test_df[extended_features],
    "engineered": custom_features(test_df)
}

y_train = train_df["isFraud"]
y_val = val_df["isFraud"]
y_test = test_df["isFraud"]

In [8]:
def get_preprocessor(X):
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()

    num_cols = X.select_dtypes(include=['number']).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))])


    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)])

    return preprocessor

In [9]:
def create_pipeline(X):
    preprocessor = get_preprocessor(X)

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear"))])

    return pipe

In [10]:
def evaluate_model(model, X, y):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:,1]

    return {"pr_auc": average_precision_score(y, probabilities),
            "roc_auc": roc_auc_score(y, probabilities),
            "precision": precision_score(y, predictions, zero_division=0),
            "recall": recall_score(y, predictions, zero_division=0),
            "f1": f1_score(y, predictions, zero_division=0)
            }

In [11]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud-detection-baseline")

results = []

for name in feature_sets:

    with mlflow.start_run(run_name=f'logistic_{name}'):
    
        X_train = X_train_sets[name]
        X_val = X_val_sets[name]
    
        pipe = create_pipeline(X_train)
    
        pipe.fit(X_train, y_train)
    
        metrics = evaluate_model(pipe, X_val, y_val)
    
        mlflow.log_param("dataset", name)

        mlflow.log_param("model", "logistic_regression")

        mlflow.log_param("class_weight", "balanced")

        mlflow.log_param("feature_count", X_train.shape[1])

        mlflow.log_metrics(metrics)

        results.append({"dataset": name, **metrics})

🏃 View run logistic_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/05412190188841afa46fbad5029bd15c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run logistic_transaction at: http://127.0.0.1:5000/#/experiments/1/runs/0599f18cc52b4635afdff4e5e38b0d80
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run logistic_engineered at: http://127.0.0.1:5000/#/experiments/1/runs/4bf7c15e654744199d56dd55e049450d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [12]:
results_df = pd.DataFrame(results)
results_df.sort_values("pr_auc", ascending=False)

,dataset,pr_auc,roc_auc,precision,recall,f1
2,engineered,0.155155,0.765067,0.075197,0.708416,0.135962
1,transaction,0.143958,0.751313,0.077953,0.675542,0.139777
0,baseline,0.119459,0.732055,0.082960,0.615385,0.146210


In [13]:
X_train_sets['engineered'].shape

(413378, 21)